# Inverse FX Chain Predictor — Gradio test app (effect TYPE + ORDER + PARAMETERS)

Pick a **test** pair (dry + wet audio from songs the model never saw), run the three-head model, and compare
1. **which effects** were applied, 2. **in which order**, and 3. **with which settings** (12 parameters)
with the **actual** values underneath. A second tab lets you upload your own dry/wet files.

The model predicts one of **12 audible chains** — EQ and Reverb commute, so e.g. *EQ → Reverb → Compressor* and
*Reverb → EQ → Compressor* sound the same and are both shown as **{EQ, Reverb} → Compressor**.

**Kaggle settings:** Internet **ON** · Add-ons → Secrets → `HF_TOKEN` (read access is enough) · Accelerator: *None* is fine.

**Parameter view:** every parameter is drawn on its range — filled dot = model, ring = actual value, shaded band =
tolerance (e.g. EQ frequency ±0.5 octave, gain ±1.5 dB, ratio within ×1.5). Parameters are compared only for effects that
are in **both** the predicted and the actual chain.

**Run:** *Run All*, then open the **public gradio.live link** printed in Section 8 (Kaggle cannot show the app inline).
Run Section 9 to stop it.

## 1. Setup

In [4]:
import os, sys, json, time, shutil, importlib, subprocess
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"

if importlib.util.find_spec("gradio") is None or int(importlib.import_module("gradio").__version__.split(".")[0]) < 5:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio>=5.9"])
for mod, spec in {"datasets": "datasets>=2.19", "soundfile": "soundfile", "librosa": "librosa"}.items():
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec])

import numpy as np, pandas as pd, torch, soundfile as sf, librosa, gradio as gr
from matplotlib.figure import Figure
print(f"torch {torch.__version__} | gradio {gr.__version__} | Python {sys.version.split()[0]}")

torch 2.11.0+cu128 | gradio 6.26.0 | Python 3.13.15


## 2. Configuration

In [5]:
MODEL_REPO = "APINAJA/inverse-fx-order-params-logmel-transformer"   # three-head model
DATASET_ID = "APINAJA/inverse-fx-phase1-effect-order"
DATASET_REVISION = "a5d0eeff7f8d9062b30f1971a2bed342e0de2521"   # same dataset version the model was trained on
NUM_DEMO_PAIRS = 100             # spread as evenly as possible over the 12 audible chains (8-9 each)
MAX_PAIRS_PER_SONG = 6           # 22 test songs x 6 >= 100 -> examples come from many songs
MAX_ROWS_TO_SCAN = 990           # whole test split at most (streamed, never fully downloaded)
REBUILD_DEMO = False             # True = pick the demo pairs again

WORK = "/kaggle/working/fx_params_demo" if os.path.exists("/kaggle/working") else os.path.abspath("fx_params_demo")
MODEL_DIR, DEMO_DIR = os.path.join(WORK, "model_repo"), os.path.join(WORK, "demo_data")
os.makedirs(DEMO_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| work dir:", WORK)

device: cuda | work dir: /kaggle/working/fx_params_demo


## 3. Hugging Face login
Token from the Kaggle secret `HF_TOKEN` (never printed).

In [6]:
_tok = None
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
_tok = _tok or os.environ.get("HF_TOKEN")
if _tok:
    os.environ["HF_TOKEN"] = _tok
    from huggingface_hub import HfApi
    print("Logged in to Hugging Face as:", HfApi(token=_tok).whoami().get("name"))
else:
    print("WARNING: no HF_TOKEN — a private model repo cannot be downloaded.")
del _tok

Logged in to Hugging Face as: APINAJA


## 4. Load the trained model
Downloads `final/` (weights + config), `fxp3/` (the exact model code used in training) and `results/test_predictions.csv` (Kaggle test predictions, used to verify this app). The parameter ranges verified from the data during training (`PARAM_RANGES`) are applied, so values are shown in the same units as in the training report.

In [7]:
from huggingface_hub import snapshot_download
snapshot_download(MODEL_REPO, repo_type="model", local_dir=MODEL_DIR, token=os.environ.get("HF_TOKEN"),
                  allow_patterns=["fxp3/*", "final/*", "results/test_predictions.csv"])
for m in [m for m in sys.modules if m == "fxp3" or m.startswith("fxp3.")]:
    del sys.modules[m]                       # re-runs pick up the downloaded code
sys.path.insert(0, MODEL_DIR)

from fxp3.model import build_model, count_parameters
from fxp3.audio import AudioPreprocessor
from fxp3.metrics import decode_chains, param_error
from fxp3.common import (EFFECTS, ORDER_QUESTIONS, ORDER_KEYS, CHAINS12, CHAIN12_NAMES, CHAINS15, targets_from_positions,
                         PARAM_SPEC, PARAM_KEYS, PARAM_NAMES, PARAM_EFFECT_INDEX, N_PARAMS, apply_param_config,
                         param_vector, denormalize)

CFG = json.load(open(os.path.join(MODEL_DIR, "final", "config.json")))["experiment_config"]
apply_param_config(CFG)                      # data-verified ranges / key names from training
model = build_model(CFG)
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, "final", "model.pt"), map_location="cpu"))
model.to(DEVICE).eval()
PRE = AudioPreprocessor(CFG)
THRESHOLD = float(CFG["THRESHOLD"])
SR, N_SAMPLES = PRE.sample_rate, PRE.num_samples
POS_COLS = CFG["POSITION_COLUMNS"]

state = json.load(open(os.path.join(MODEL_DIR, "final", "training_state.json")))
tm = json.load(open(os.path.join(MODEL_DIR, "final", "test_metrics.json")))
print(f"Model loaded: {count_parameters(model):,} params | best epoch {state['best_epoch']} | fusion {CFG['FUSION']}")
print(f"Kaggle test results: 12-chain accuracy {tm['chain12_accuracy']:.3f} | order accuracy {tm['order/accuracy']:.3f} "
      f"(EQ<Comp {tm['eq_before_comp/accuracy']:.3f}, Rev<Comp {tm['rev_before_comp/accuracy']:.3f}) | "
      f"presence macro F1 {tm['presence/macro_f1']:.3f}")
print(f"                    parameters: MAE {tm['param/mae_norm']:.4f} (mean baseline {tm['param/mae_norm_baseline']:.4f}) | "
      f"skill {tm['param/skill']:.3f} | within tolerance {tm['param/within_tol']:.1%}")
display(pd.DataFrame(tm["param_table"]).set_index("name")[["support", "mae_norm", "error", "within_tol", "r2"]]
        .rename(columns={"support": "n", "mae_norm": "MAE (0-1)", "error": "typical error*", "within_tol": "within tol."})
        .style.format({"MAE (0-1)": "{:.4f}", "typical error*": "{:.3f}", "within tol.": "{:.1%}", "r2": "{:.3f}"}))
print("* typical error: dB / unit for linear params, octaves for EQ frequency, ×factor for Q, ratio, attack, release")

Model loaded: 2,309,137 params | best epoch 21 | fusion concat_diff
Kaggle test results: 12-chain accuracy 0.539 | order accuracy 0.663 (EQ<Comp 0.568, Rev<Comp 0.758) | presence macro F1 0.932
                    parameters: MAE 0.2133 (mean baseline 0.2541) | skill 0.160 | within tolerance 41.2%


,n,MAE (0-1),typical error*,within tol.,r2
name,,,,,
EQ frequency,726,0.1508,0.953,38.7%,0.442
EQ gain,726,0.1771,4.250,24.1%,0.495
EQ Q,726,0.1954,1.501,59.8%,0.233
Comp threshold,726,0.1398,3.495,51.2%,0.572
Comp ratio,726,0.2205,1.426,60.5%,0.149
Comp attack,726,0.2266,2.427,44.5%,0.050
Comp release,726,0.2571,1.808,35.1%,-0.051
Comp makeup,726,0.2305,1.383,37.6%,0.113
Rev room size,726,0.2245,0.135,39.1%,0.048


* typical error: dB / unit for linear params, octaves for EQ frequency, ×factor for Q, ratio, attack, release


## 5. Inference helpers
Same preprocessing as training (`fxp3.audio.AudioPreprocessor`: mono, 44.1 kHz, first 10 s / zero-pad, float32, no normalisation). Ground-truth metadata is never passed to the model.

In [8]:
@torch.no_grad()
def predict(dry_wave, wet_wave):
    d = torch.from_numpy(np.ascontiguousarray(dry_wave))[None].to(DEVICE)
    w = torch.from_numpy(np.ascontiguousarray(wet_wave))[None].to(DEVICE)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
        out = model(d, w)                        # {"presence": [1,3], "order": [1,2], "params": [1,12]}
    pres = torch.sigmoid(out["presence"].float())[0].cpu().numpy()
    order = torch.sigmoid(out["order"].float())[0].cpu().numpy()
    params = out["params"].float()[0].cpu().numpy()   # already sigmoid -> normalised 0-1
    dec = decode_chains(pres[None], order[None])  # joint probability of the 12 audible chains -> argmax
    c = int(dec["pred12"][0])
    return {"presence": pres, "order": order, "pred_presence": (pres >= THRESHOLD).astype(int),
            "pred_order": (order >= 0.5).astype(int), "chain12": c, "chain_name": CHAIN12_NAMES[c],
            "chain_prob": float(dec["chain12_probs"][0, c]), "chain_probs": dec["chain12_probs"][0],
            "params": params, "params_real": [float(denormalize(j, params[j])) for j in range(N_PARAMS)]}


def load_file_for_model(path):
    """Decode exactly like training (soundfile) when possible; librosa fallback for mp3 etc."""
    try:
        return PRE({"path": path, "bytes": None})
    except Exception:
        arr, sr = librosa.load(path, sr=None, mono=False)
        return PRE({"array": np.atleast_2d(arr).astype(np.float32), "sampling_rate": sr})


def spectrogram_figure(dry_wave, wet_wave, title=""):
    with torch.no_grad():
        lm = model.logmel.raw_log_mel(torch.from_numpy(np.stack([dry_wave, wet_wave])).to(DEVICE)).cpu().numpy()
    diff = lm[1] - lm[0]
    fig = Figure(figsize=(15, 3.3), dpi=90)
    axes = fig.subplots(1, 3, sharey=True)
    lo, hi = np.percentile(lm, 5), lm.max()
    ext = [0, CFG["AUDIO_LENGTH_SECONDS"], 0, CFG["N_MELS"]]
    for ax, m, t in zip(axes[:2], lm, ["Dry log-mel", "Wet log-mel"]):
        ax.imshow(m, origin="lower", aspect="auto", cmap="magma", vmin=lo, vmax=hi, extent=ext)
        ax.set_title(t, loc="left", fontsize=10)
    lim = float(np.percentile(np.abs(diff), 99)) or 1.0
    im = axes[2].imshow(diff, origin="lower", aspect="auto", cmap="RdBu_r", vmin=-lim, vmax=lim, extent=ext)
    axes[2].set_title("Wet − Dry (red = louder in wet, blue = quieter)", loc="left", fontsize=10)
    fig.colorbar(im, ax=axes[2], fraction=0.04, pad=0.02, label="dB-like")
    for ax in axes:
        ax.set_xlabel("Time (s)", fontsize=9); ax.tick_params(labelsize=8)
    axes[0].set_ylabel("Mel bin", fontsize=9)
    if title:
        fig.suptitle(title, x=0.01, ha="left", fontsize=10)
    fig.tight_layout()
    return fig


def _yn(v, yes="present", no="absent"):
    return yes if int(v) else no


def result_markdown(res, truth=None):
    pc = CHAINS12[res["chain12"]]                 # (name, short, presence, o1, o2, canonical)
    out = ["### Model prediction", f"**Predicted chain: {res['chain_name']}**  (probability {res['chain_prob']:.2f})", "",
           "| Effect | Probability | Predicted |", "|---|---|---|"]
    out += [f"| {e} | {res['presence'][k]:.3f} | {_yn(res['pred_presence'][k])} |" for k, e in enumerate(EFFECTS)]
    out += ["", "| Order question | P(yes) | Order head says | Used in predicted chain? |", "|---|---|---|---|"]
    for j, q in enumerate(ORDER_QUESTIONS):
        used = pc[3 + j] is not None
        out.append(f"| {q}? | {res['order'][j]:.3f} | {_yn(res['pred_order'][j], 'yes', 'no')} | "
                   f"{'yes' if used else 'no (needed only when both effects are present)'} |")
    out += ["", "| Parameter (predicted effects) | Model |", "|---|---|"]
    out += [f"| {PARAM_NAMES[j]} | {fmt_value(j, res['params_real'][j])} |" for j in range(N_PARAMS)
            if res["pred_presence"][PARAM_EFFECT_INDEX[j]]]
    if truth is None:
        return "\n".join(out)
    t = truth["targets"]
    chain_ok = res["chain12"] == t["class12"]
    tc = CHAINS12[t["class12"]]
    out += ["", "---", "### Actual (ground truth)",
            f"**True chain as rendered: {' → '.join(truth['chain'])}**",
            f"**Audible chain: {tc[0]}**"]
    if tc[2][0] and tc[2][2] and tc[3] == tc[4]:     # EQ and Reverb on the same side of the compressor (or no compressor)
        out.append("_EQ and Reverb are both linear, so their order relative to each other cannot be heard; "
                   "both renders count as this audible chain._")
    out += ["", "| Effect | Actual | Predicted | Result |", "|---|---|---|---|"]
    for k, e in enumerate(EFFECTS):
        ok = int(res["pred_presence"][k]) == int(t["presence"][k])
        out.append(f"| {e} | {_yn(t['presence'][k])} | {_yn(res['pred_presence'][k])} | {'✅ correct' if ok else '❌ wrong'} |")
    out += ["", "| Order question | Actual | Order head says | Result |", "|---|---|---|---|"]
    for j, q in enumerate(ORDER_QUESTIONS):
        if t["order_mask"][j]:
            ok = int(res["pred_order"][j]) == int(t["order"][j])
            out.append(f"| {q}? | {_yn(t['order'][j], 'yes', 'no')} | {_yn(res['pred_order'][j], 'yes', 'no')} | "
                       f"{'✅ correct' if ok else '❌ wrong'} |")
        else:
            out.append(f"| {q}? | — (not both present) | {_yn(res['pred_order'][j], 'yes', 'no')} | — |")
    out += ["", f"**Chain correct: {'✅ yes' if chain_ok else '❌ no'}**", "",
            "**Parameters** (actual settings come from the dataset — never given to the model)", "",
            "| Parameter | Actual | Model | Error | Within tolerance |", "|---|---|---|---|---|"]
    for j in range(N_PARAMS):
        if not truth["param_mask"][j]:
            continue
        e_txt, ok = fmt_error(j, res["params"][j], truth["param_true"][j])
        used = bool(res["pred_presence"][PARAM_EFFECT_INDEX[j]])
        out.append(f"| {PARAM_NAMES[j]} | {fmt_value(j, denormalize(j, truth['param_true'][j]))} | "
                   f"{fmt_value(j, res['params_real'][j])}{'' if used else ' (effect predicted absent)'} | {e_txt} | "
                   f"{('✅' if ok else '❌') if used else '—'} ({tolerance_text(j)}) |")
    if truth.get("low_effect_change") not in (None, "", "[]"):
        out.append(f"- ⚠️ Flagged as a *low* (subtle) effect change in the dataset: {truth['low_effect_change']}")
    out.append(f"\n<sub>{truth['sample_id']} · {truth['source_category']} · {truth['source_song']}</sub>")
    return "\n".join(out)


def probs_for_label(res):
    """Top chains for the gr.Label bar chart."""
    return {CHAIN12_NAMES[i]: float(res["chain_probs"][i]) for i in range(len(CHAIN12_NAMES))}

import html as _html

import numpy as np

from fxp3.common import EFFECTS, PARAM_EFFECT_INDEX, PARAM_SPEC, denormalize, normalize
from fxp3.metrics import param_error

# Effect identity colours: fixed categorical slots 1-3 (blue, orange, aqua) — marks only, never text.
FX_COLOR = {"EQ": "#2a78d6", "Compressor": "#eb6834", "Reverb": "#1baf7a"}
GOOD, BAD = "#1baf7a", "#e34948"
_T = "var(--body-text-color, #1f1f1f)"
_T2 = "var(--body-text-color-subdued, #6b6b6b)"
_LINE = "var(--border-color-primary, #d9d9d6)"
_SURF = "var(--block-background-fill, transparent)"


def _esc(x):
    return _html.escape(str(x))


# ----------------------------------------------------------------------------- parameter formatting
def fmt_value(i, v):
    """Physical value of parameter i as short text with unit."""
    _, k, _, unit, *_ = PARAM_SPEC[i]
    v = float(v)
    if unit == "Hz":
        return f"{v / 1000:.2f} kHz" if v >= 1000 else f"{v:.0f} Hz"
    if unit == "dB":
        return f"{v:+.1f} dB" if k == "gain" else f"{v:.1f} dB"
    if unit == ":1":
        return f"{v:.1f}:1"
    if unit == "ms":
        return f"{v:.1f} ms" if v < 10 else f"{v:.0f} ms"
    return f"{v:.2f}"


def fmt_error(i, pred_norm, true_norm):
    """(text, within tolerance) for one prediction."""
    _, _, _, unit, _, _, _, tol, kind = PARAM_SPEC[i]
    e = param_error(i, np.array([pred_norm]), np.array([true_norm]))
    err, ok = float(e["err"][0]), bool(e["ok"][0])
    if kind == "octave":
        return f"{err:.2f} oct", ok
    if kind == "factor":
        return f"×{np.exp(err):.2f}", ok
    return (f"{err:.1f} {unit}" if unit else f"{err:.2f}"), ok


def tolerance_text(i):
    _, _, _, unit, _, _, _, tol, kind = PARAM_SPEC[i]
    if kind == "octave":
        return f"±{tol:g} octave"
    if kind == "factor":
        return f"within ×{tol:g}"
    return f"±{tol:g} {unit}".strip()


def tolerance_band(i, true_norm):
    """[lo, hi] on the 0-1 track that counts as 'within tolerance' around the true value."""
    _, _, _, _, _, lo, hi, tol, kind = PARAM_SPEC[i]
    v = float(denormalize(i, true_norm))
    if kind == "octave":
        a, b = v / 2 ** tol, v * 2 ** tol
    elif kind == "factor":
        a, b = v / tol, v * tol
    else:
        a, b = v - tol, v + tol
    a, b = max(lo, min(hi, a)), max(lo, min(hi, b))
    return float(np.clip(normalize(i, a), 0, 1)), float(np.clip(normalize(i, b), 0, 1))


def effect_summary(effect, params_norm):
    """One-line settings of an effect, e.g. '1.20 kHz · +4.5 dB · Q 1.30'."""
    idx = [j for j, p in enumerate(PARAM_SPEC) if p[0] == effect]
    v = {PARAM_SPEC[j][1]: fmt_value(j, denormalize(j, params_norm[j])) for j in idx}
    if effect == "EQ":
        return f"{v['frequency']} · {v['gain']} · Q {v['q']}"
    if effect == "Compressor":
        return f"{v['threshold']} · {v['ratio']} · {v['attack']} / {v['release']} · makeup {v['makeup_gain']}"
    return f"room {v['room_size']} · damp {v['damping']} · wet {v['wet_mix']} · dry {v['dry_mix']}"


# ----------------------------------------------------------------------------- building blocks
def _chip(effect, faded=False, sub=None):
    c = FX_COLOR[effect]
    op = "0.35" if faded else "1"
    sub_html = (f'<span style="display:block;font-weight:500;font-size:11.5px;color:{_T2};margin-top:2px;'
                f'white-space:nowrap;font-variant-numeric:tabular-nums">{_esc(sub)}</span>') if sub else ""
    return (f'<span style="display:inline-flex;align-items:flex-start;gap:6px;padding:6px 12px;border-radius:8px;'
            f'border:1.5px solid {c};background:{c}22;color:{_T};font-weight:600;font-size:14px;opacity:{op}">'
            f'<span style="width:9px;height:9px;border-radius:50%;background:{c};margin-top:5px;flex:none"></span>'
            f'<span>{_esc(effect)}{sub_html}</span></span>')


def _node(text):
    return (f'<span style="padding:5px 10px;border-radius:999px;border:1px solid {_LINE};color:{_T2};'
            f'font-size:12px;white-space:nowrap">{_esc(text)}</span>')


_ARROW = f'<span style="color:{_T2};font-size:18px;padding:0 2px">→</span>'


def chain_steps(c12):
    """CHAINS12 entry -> list of steps; a step is an effect name or ("EQ", "Reverb") for an unordered group."""
    _, _, pres, o1, o2, canon = c12
    if pres[0] and pres[2] and o1 == o2:                 # EQ and Reverb on the same side -> order not audible
        group = ("EQ", "Reverb")
        if not pres[1]:
            return [group]
        return [group, "Compressor"] if o1 == 1 else ["Compressor", group]
    return list(canon)


def chain_diagram(c12, settings=None):
    """settings: {effect: one-line parameter text} shown inside each chip (optional)."""
    settings = settings or {}
    parts = [_node("dry audio")]
    for step in chain_steps(c12):
        parts.append(_ARROW)
        if isinstance(step, tuple):
            parts.append(f'<span style="display:inline-flex;flex-direction:column;gap:4px;padding:6px;border-radius:10px;'
                         f'border:1.5px dashed {_LINE};align-items:center">'
                         f'<span style="display:flex;gap:6px;flex-wrap:wrap">{_chip(step[0], sub=settings.get(step[0]))}'
                         f'{_chip(step[1], sub=settings.get(step[1]))}</span>'
                         f'<span style="font-size:11px;color:{_T2}">any order (not audible)</span></span>')
        else:
            parts.append(_chip(step, sub=settings.get(step)))
    parts += [_ARROW, _node("wet audio")]
    return f'<div style="display:flex;flex-wrap:wrap;align-items:center;gap:6px;margin:6px 0 2px">{"".join(parts)}</div>'


def _bar(p, color, threshold=0.5):
    w = max(0.0, min(1.0, float(p))) * 100
    return (f'<div style="position:relative;height:12px;border-radius:6px;background:{_LINE};min-width:120px">'
            f'<div style="width:{w:.1f}%;height:100%;border-radius:6px;background:{color}"></div>'
            f'<div title="decision threshold {threshold}" style="position:absolute;left:{threshold * 100:.0f}%;top:-3px;'
            f'width:2px;height:18px;background:{_T}"></div></div>')


def _mark(ok, yes="correct", no="wrong"):
    if ok is None:
        return f'<span style="color:{_T2}">—</span>'
    c, sym, txt = (GOOD, "✓", yes) if ok else (BAD, "✗", no)
    return (f'<span style="display:inline-flex;align-items:center;gap:4px;font-weight:600;color:{_T};white-space:nowrap">'
            f'<span style="display:inline-grid;place-items:center;width:18px;height:18px;border-radius:50%;'
            f'background:{c};color:#fff;font-size:12px;flex:none">{sym}</span>{txt}</span>')


def _section(title, body, sub=""):
    sub_html = f'<div style="font-size:12px;color:{_T2};margin-top:2px">{sub}</div>' if sub else ""
    return (f'<div style="border:1px solid {_LINE};border-radius:12px;padding:14px 16px;margin:10px 0;background:{_SURF}">'
            f'<div style="font-size:13px;font-weight:700;letter-spacing:.02em;text-transform:uppercase;color:{_T2}">{title}</div>'
            f'{sub_html}{body}</div>')


def _track(i, pred_norm, true_norm=None, color="#888", faded=False):
    """Range track lo..hi: tolerance band + true value (ring) + prediction (filled dot)."""
    _, _, _, _, scale, lo, hi, _, _ = PARAM_SPEC[i]
    band = ""
    if true_norm is not None:
        a, b = tolerance_band(i, true_norm)
        band = (f'<div title="within tolerance" style="position:absolute;left:{a * 100:.1f}%;width:{max(0.6, (b - a) * 100):.1f}%;'
                f'top:3px;height:8px;border-radius:4px;background:{color}40"></div>'
                f'<div title="actual" style="position:absolute;left:calc({true_norm * 100:.1f}% - 7px);top:0;width:10px;height:10px;'
                f'border-radius:50%;border:2px solid {_T};background:var(--background-fill-primary, #fff)"></div>')
    dot = ("" if pred_norm is None else
           f'<div title="predicted" style="position:absolute;left:calc({pred_norm * 100:.1f}% - 6px);top:1px;width:12px;height:12px;'
           f'border-radius:50%;background:{color};box-shadow:0 0 0 2px var(--background-fill-primary, #fff)"></div>')
    ends = (f'<div style="display:flex;justify-content:space-between;font-size:10.5px;color:{_T2};margin-top:3px;'
            f'font-variant-numeric:tabular-nums"><span>{_esc(fmt_value(i, lo))}</span>'
            f'<span>{"log scale" if scale == "log" else ""}</span><span>{_esc(fmt_value(i, hi))}</span></div>')
    return (f'<div style="opacity:{0.5 if faded else 1}"><div style="position:relative;height:14px;min-width:160px">'
            f'<div style="position:absolute;left:0;right:0;top:6px;height:2px;border-radius:1px;background:{_LINE}"></div>'
            f'{band}{dot}</div>{ends}</div>')


def params_section(res, truth=None):
    """Per-effect parameter tables: shown for effects in the predicted OR the actual chain."""
    pred_pres = [int(x) for x in res["pred_presence"]]
    true_pres = [int(x) for x in truth["targets"]["presence"]] if truth is not None else [0, 0, 0]
    p_true = truth.get("param_true") if truth is not None else None
    p_mask = truth.get("param_mask") if truth is not None else None
    groups, n_ok, n_tot, maes = [], 0, 0, []
    widths = [13, 37, 15, 15, 12, 8] if truth is not None else [15, 55, 30]   # same columns for every effect
    cols = "<colgroup>" + "".join(f'<col style="width:{w}%">' for w in widths) + "</colgroup>"
    for k, e in enumerate(EFFECTS):
        if not (pred_pres[k] or true_pres[k]):
            continue
        if truth is None:
            status = "predicted settings"
        elif pred_pres[k] and true_pres[k]:
            status = "in both chains — compare with the actual settings"
        elif pred_pres[k]:
            status = "model says present, but it is NOT in the actual chain — no actual settings"
        else:
            status = "in the actual chain, but the model says absent — its values are shown faded for reference"
        faded = not pred_pres[k]
        rows = []
        for j in [j for j in range(len(PARAM_SPEC)) if PARAM_EFFECT_INDEX[j] == k]:
            pn = float(res["params"][j])
            has_true = p_true is not None and p_mask is not None and p_mask[j] and true_pres[k]
            tn = float(p_true[j]) if has_true else None
            err_txt, ok = fmt_error(j, pn, tn) if has_true else ("", None)
            if has_true and not faded:
                n_ok += int(ok); n_tot += 1; maes.append(abs(pn - tn))
            cells = (f'<td style="padding:7px 8px 7px 0;color:{_T};white-space:nowrap;opacity:{0.6 if faded else 1}">'
                     f'{_esc(PARAM_SPEC[j][2].split(" ", 1)[1] if " " in PARAM_SPEC[j][2] else PARAM_SPEC[j][2])}</td>'
                     f'<td style="padding:7px 10px">{_track(j, pn, tn, FX_COLOR[e], faded)}</td>'
                     f'<td style="padding:7px 8px;color:{_T};white-space:nowrap;font-variant-numeric:tabular-nums;opacity:{0.6 if faded else 1}">'
                     f'model <b>{_esc(fmt_value(j, denormalize(j, pn)))}</b></td>')
            if truth is not None:
                cells += (f'<td style="padding:7px 8px;color:{_T};white-space:nowrap;font-variant-numeric:tabular-nums">'
                          f'actual <b>{_esc(fmt_value(j, denormalize(j, tn))) if has_true else "—"}</b></td>'
                          f'<td style="padding:7px 8px;color:{_T2};white-space:nowrap;font-variant-numeric:tabular-nums">'
                          f'{("off by " + _esc(err_txt)) if has_true else ""}</td>'
                          f'<td style="padding:7px 0;white-space:nowrap" title="tolerance {tolerance_text(j)}">'
                          f'{_mark(ok, "close", "far") if has_true and not faded else ""}</td>')
            rows.append(f"<tr>{cells}</tr>")
        groups.append(f'<div style="margin-top:12px"><div style="display:flex;align-items:center;gap:10px;flex-wrap:wrap">'
                      f'{_chip(e, faded=faded)}<span style="font-size:12px;color:{_T2}">{_esc(status)}</span></div>'
                      f'<table style="width:100%;border-collapse:collapse;margin-top:4px;table-layout:fixed">{cols}{"".join(rows)}</table></div>')
    if not groups:
        return ""
    if truth is not None and n_tot:
        sub = (f"<b style='color:{_T}'>{n_ok} of {n_tot}</b> parameters within tolerance · mean error "
               f"{np.mean(maes):.3f} on the 0–1 scale · filled dot = model, ring = actual, shaded = tolerance "
               f"(hover ✓/✗ to see it)")
    elif truth is not None:
        sub = "no effect is in both chains, so no parameter can be compared · filled dot = model"
    else:
        sub = "filled dot = model prediction on the parameter's range"
    return _section("Effect parameters", "".join(groups), sub)


# ----------------------------------------------------------------------------- full card
def result_html(res, truth=None, chains12=None, effects=None, questions=None, threshold=0.5):
    pc = chains12[res["chain12"]]
    blocks = []
    pred_set = {e: effect_summary(e, res["params"]) for k, e in enumerate(effects) if pc[2][k]}
    if truth is not None:                                 # verdict banner (status colour + icon + words)
        t = truth["targets"]
        ok = res["chain12"] == t["class12"]
        c = GOOD if ok else BAD
        msg = ("Correct — the model found the actual effect chain." if ok else
               f"Wrong — predicted <b>{_esc(pc[0])}</b>, actual <b>{_esc(chains12[t['class12']][0])}</b>.")
        blocks.append(f'<div style="display:flex;align-items:center;gap:10px;padding:12px 16px;border-radius:12px;'
                      f'border:1.5px solid {c};background:{c}1f;color:{_T};font-size:15px">'
                      f'<span style="display:inline-grid;place-items:center;width:26px;height:26px;border-radius:50%;'
                      f'background:{c};color:#fff;font-weight:700;flex:none">{"✓" if ok else "✗"}</span><span>{msg}</span></div>')

    # chain diagrams with the settings inside each effect
    pred_body = (chain_diagram(pc, pred_set) +
                 f'<div style="font-size:13px;color:{_T2}">confidence {res["chain_prob"]:.0%} (probability of this chain)</div>')
    blocks.append(_section("Model prediction", pred_body))
    if truth is not None:
        t = truth["targets"]
        tc = chains12[t["class12"]]
        true_set = {}
        if truth.get("param_true") is not None:
            true_set = {e: effect_summary(e, truth["param_true"]) for k, e in enumerate(effects) if tc[2][k]}
        rendered = " → ".join(truth["chain"])
        blocks.append(_section("Ground truth", chain_diagram(tc, true_set), f"rendered in the dataset as: {_esc(rendered)}"))

    # presence bars
    rows = []
    for k, e in enumerate(effects):
        p = float(res["presence"][k])
        pred = "present" if res["pred_presence"][k] else "absent"
        actual, ok = "", None
        if truth is not None:
            a = int(truth["targets"]["presence"][k])
            actual = "present" if a else "absent"
            ok = int(res["pred_presence"][k]) == a
        rows.append(f'<tr><td style="padding:6px 8px 6px 0">{_chip(e, faded=not res["pred_presence"][k])}</td>'
                    f'<td style="padding:6px 8px;width:45%">{_bar(p, FX_COLOR[e], threshold)}</td>'
                    f'<td style="padding:6px 8px;color:{_T};font-variant-numeric:tabular-nums">{p:.0%}</td>'
                    f'<td style="padding:6px 8px;color:{_T};white-space:nowrap">model: <b>{pred}</b></td>'
                    + (f'<td style="padding:6px 8px;color:{_T};white-space:nowrap">actual: <b>{actual}</b></td><td style="padding:6px 0">{_mark(ok)}</td>'
                       if truth is not None else "") + "</tr>")
    blocks.append(_section("Which effects are present?", f'<table style="width:100%;border-collapse:collapse;margin-top:6px">{"".join(rows)}</table>',
                           "bar = model probability · tick = 50 % decision threshold"))

    # order questions
    rows = []
    pairs = [("EQ", "Compressor"), ("Reverb", "Compressor")]
    for j, q in enumerate(questions):
        a_eff, b_eff = pairs[j]
        p = float(res["order"][j])
        used = pc[3 + j] is not None
        defined = bool(truth["targets"]["order_mask"][j]) if truth is not None else None
        relevant = used or bool(defined)
        model_ans = "before" if res["pred_order"][j] else "after"
        if truth is not None and defined:
            act = "before" if truth["targets"]["order"][j] else "after"
            ok = int(res["pred_order"][j]) == int(truth["targets"]["order"][j])
        elif truth is not None:
            act, ok = "n/a", None
        else:
            act, ok = "", None
        dim = "" if relevant else "opacity:.45;"
        note = "" if relevant else f'<div style="font-size:11px;color:{_T2}">not needed — both effects are not present</div>'
        rows.append(f'<tr style="{dim}"><td style="padding:8px 8px 8px 0;color:{_T};white-space:nowrap"><b>{_esc(q)}?</b>{note}</td>'
                    f'<td style="padding:8px;width:38%">{_bar(p, FX_COLOR[a_eff])}'
                    f'<div style="display:flex;justify-content:space-between;font-size:11px;color:{_T2};margin-top:3px">'
                    f'<span>after</span><span>before</span></div></td>'
                    f'<td style="padding:8px;color:{_T};font-variant-numeric:tabular-nums">{p:.0%}</td>'
                    f'<td style="padding:8px;color:{_T};white-space:nowrap">model: <b>{_esc(model_ans)}</b></td>'
                    + (f'<td style="padding:8px;color:{_T};white-space:nowrap">actual: <b>{_esc(act)}</b></td><td style="padding:8px 0">{_mark(ok)}</td>'
                       if truth is not None else "") + "</tr>")
    blocks.append(_section("Order — where does the Compressor sit?",
                           f'<table style="width:100%;border-collapse:collapse;margin-top:6px">{"".join(rows)}</table>',
                           "EQ ↔ Reverb order is not audible, so only each effect's position relative to the Compressor is predicted"))

    blocks.append(params_section(res, truth))
    return f'<div style="font-family:inherit;max-width:1100px">{"".join(blocks)}</div>'


def result_view(res, truth=None):
    return result_html(res, truth, chains12=CHAINS12, effects=EFFECTS, questions=ORDER_QUESTIONS, threshold=THRESHOLD)

print("helpers ready")

helpers ready


## 6. Pick the test pairs
Streams the **test** split (no full download; ~2 GB read once, a few minutes) and keeps `NUM_DEMO_PAIRS` pairs spread evenly over the 12 audible chains, Bass / Drums / Mixture and many songs. Saved to `demo_data/` (~0.5 GB) so re-runs are instant; set `REBUILD_DEMO = True` after changing the number.

In [9]:
CATS = ["Bass", "Drums", "Mixture"]
DEMO_JSON = os.path.join(DEMO_DIR, "demo.json")


def _targets(row):
    return targets_from_positions(tuple(int(row[c]) for c in POS_COLS))


def chain_quota(total, n_cls):
    """Even split: e.g. 100 over 12 chains -> 4 chains x 9 + 8 chains x 8."""
    return {c: total // n_cls + (1 if c < total % n_cls else 0) for c in range(n_cls)}


def select_rows(rows, total=NUM_DEMO_PAIRS, max_per_song=MAX_PAIRS_PER_SONG, max_rows=MAX_ROWS_TO_SCAN):
    """Deterministic: slot s of audible chain c prefers category CATS[(c + s) % 3]; <= max_per_song picks per song."""
    n_cls = len(CHAINS12)
    quota = chain_quota(total, n_cls)
    picked = {c: [] for c in range(n_cls)}
    backup = {c: [] for c in range(n_cls)}
    per_song = {}
    for n, row in enumerate(rows):
        if n >= max_rows:
            break
        c = _targets(row)["class12"]
        slot = len(picked[c])
        if slot < quota[c] and row["source_category"] == CATS[(c + slot) % 3] \
                and per_song.get(row["source_song"], 0) < max_per_song:
            picked[c].append(row)
            per_song[row["source_song"]] = per_song.get(row["source_song"], 0) + 1
        elif len(backup[c]) < quota[c]:
            backup[c].append(row)
        if all(len(picked[c]) == quota[c] for c in range(n_cls)):
            break
    for c in range(n_cls):                           # fallback if the preferred spread was not found
        for row in backup[c]:
            if len(picked[c]) < quota[c] and all(row is not r for r in picked[c]):
                picked[c].append(row)
    # interleave chains so neighbouring pairs in the list are different chains
    out, k = [], 0
    while any(picked.values()):
        c = k % n_cls
        if picked[c]:
            out.append(picked[c].pop(0))
        k += 1
    return out


def _write_pair(i, row):
    base = os.path.join(DEMO_DIR, f"pair{i:02d}")
    files = {}
    for kind in ("dry", "wet"):
        val = row[f"{kind}_audio"]
        ext = os.path.splitext(val.get("path") or "")[1] or ".wav"
        raw = f"{base}_{kind}{ext}"                  # original bytes -> model input (exactly as in training)
        with open(raw, "wb") as f:
            f.write(val["bytes"])
        files[kind] = raw
    d, _ = sf.read(files["dry"], dtype="float32", always_2d=True)
    w, sr = sf.read(files["wet"], dtype="float32", always_2d=True)
    scale = 0.99 / max(0.99, float(np.abs(d).max()), float(np.abs(w).max()))   # same gain for both: keeps level difference
    for kind, arr in (("dry", d), ("wet", w)):
        play = f"{base}_{kind}_play.wav"             # 16-bit copy for the browser player
        sf.write(play, arr.mean(axis=1) * scale, sr, subtype="PCM_16")
        files[f"{kind}_play"] = play
    keep = ["sample_id", "source_category", "source_song", "eq_params", "compressor_params", "reverb_params",
            "parameters", "effect_change_db", "low_effect_change", *POS_COLS]
    meta = {k: row.get(k) for k in keep}
    t = _targets(row)
    p_true, has = param_vector(row.get(CFG["PARAM_TARGET_COLUMN"]), CFG.get("PARAM_KEY_MAP") or {})
    meta["param_true"] = [float(x) for x in p_true]
    meta["param_mask"] = [int(h and t["presence"][PARAM_EFFECT_INDEX[j]]) for j, h in enumerate(has)]
    meta.update(files, targets={k: (list(v) if isinstance(v, tuple) else v) for k, v in t.items()},
                chain=list(t["chain"]), chain12_name=CHAIN12_NAMES[t["class12"]],
                pair=i, display=f"Pair {i:02d} · {row['source_category']} · {row['source_song']}")
    return meta


_cached = json.load(open(DEMO_JSON)) if os.path.exists(DEMO_JSON) else None
if _cached is not None and not REBUILD_DEMO and len(_cached) == NUM_DEMO_PAIRS and all("param_true" in d for d in _cached):
    DEMO = _cached
    print(f"Loaded {len(DEMO)} cached demo pairs from {DEMO_JSON}")
else:
    if _cached is not None:
        print(f"Rebuilding the demo set ({len(_cached)} cached pairs, {NUM_DEMO_PAIRS} requested)")
    shutil.rmtree(DEMO_DIR, ignore_errors=True); os.makedirs(DEMO_DIR, exist_ok=True)
    from datasets import load_dataset, Audio
    t0 = time.time()
    stream = load_dataset(DATASET_ID, split="test", streaming=True, revision=DATASET_REVISION,
                          token=os.environ.get("HF_TOKEN"))
    stream = stream.cast_column("dry_audio", Audio(decode=False)).cast_column("wet_audio", Audio(decode=False))
    rows = select_rows(stream)
    DEMO = []
    for i, r in enumerate(rows):
        DEMO.append(_write_pair(i + 1, r))
        if (i + 1) % 25 == 0:
            print(f"  saved {i + 1}/{len(rows)} pairs")
    json.dump(DEMO, open(DEMO_JSON, "w"), indent=1)
    print(f"Selected and saved {len(DEMO)} test pairs in {time.time() - t0:.0f}s")

demo_df = pd.DataFrame([{"pair": d["pair"], "category": d["source_category"], "song": d["source_song"],
                         "true chain (rendered)": " → ".join(d["chain"]), "audible chain": d["chain12_name"]} for d in DEMO])
_miss = sum(1 for d in DEMO for j in range(N_PARAMS) if d["targets"]["presence"][PARAM_EFFECT_INDEX[j]] and not d["param_mask"][j])
print("parameter targets:", sum(sum(d["param_mask"]) for d in DEMO), "| missing for present effects:", _miss)
print(f"{len(DEMO)} pairs | {demo_df['song'].nunique()} songs | per category: {demo_df['category'].value_counts().to_dict()}")
display(demo_df.groupby("audible chain").size().reindex(CHAIN12_NAMES).rename("pairs").to_frame())

  saved 25/100 pairs
  saved 50/100 pairs
  saved 75/100 pairs
  saved 100/100 pairs
Selected and saved 100 test pairs in 60s
parameter targets: 827 | missing for present effects: 0
100 pairs | 18 songs | per category: {'Mixture': 34, 'Bass': 33, 'Drums': 33}


,pairs
audible chain,
EQ,9
Compressor,9
Reverb,9
EQ + Reverb (order not audible),9
EQ → Compressor,8
Compressor → EQ,8
Compressor → Reverb,8
Reverb → Compressor,8
"Compressor → {EQ, Reverb}",8


## 7. Verify the app against the Kaggle test run
The app's probabilities **and parameter outputs** must match `results/test_predictions.csv` (produced during training). A match proves the preprocessing here is identical to training. Small differences (≈1e-3) are normal on CPU (fp32) vs Kaggle GPU (fp16).

In [10]:
ref = pd.read_csv(os.path.join(MODEL_DIR, "results", "test_predictions.csv")).set_index("sample_id")
RESULTS = {}
rows_v = []
PROB_COLS = [f"p_{e}" for e in EFFECTS] + [f"p_{q}" for q in ORDER_KEYS]
PARAM_COLS = [f"norm_pred_{k}" for k in PARAM_KEYS]
t0 = time.time()
for n_done, d in enumerate(DEMO, 1):
    res = predict(load_file_for_model(d["dry"]), load_file_for_model(d["wet"]))
    RESULTS[d["pair"]] = res
    if n_done % 25 == 0:
        print(f"  {n_done}/{len(DEMO)} pairs predicted ({time.time() - t0:.0f}s)")
    mine = np.concatenate([res["presence"], res["order"]])
    r = ref.loc[d["sample_id"]] if d["sample_id"] in ref.index else None
    diff = float(np.max(np.abs(mine - r[PROB_COLS].to_numpy(dtype=float)))) if r is not None else np.nan
    pdiff = (float(np.max(np.abs(res["params"] - r[PARAM_COLS].to_numpy(dtype=float))))
             if r is not None and all(c in r.index for c in PARAM_COLS) else np.nan)
    flips = int(res["chain_name"] != r["pred_chain12"]) if r is not None else np.nan
    rows_v.append({"pair": d["pair"], "sample_id": d["sample_id"], "max |Δprob|": diff, "max |Δparam|": pdiff,
                   "decision flips": flips})
ver = pd.DataFrame(rows_v).set_index("pair")
display(ver.sort_values("max |Δprob|", ascending=False).head(10))   # worst 10 shown; all are checked below
max_diff = float(np.nanmax(ver[["max |Δprob|", "max |Δparam|"]].to_numpy())) if ver["max |Δprob|"].notna().any() else float("nan")
if np.isnan(max_diff):
    print("⚠️ None of the demo pairs were found in test_predictions.csv — cannot verify.")
elif max_diff < 0.02 and int(np.nansum(ver["decision flips"])) == 0:
    print(f"✅ VERIFIED: app matches the Kaggle evaluation (max difference {max_diff:.2e}).")
else:
    print(f"❌ MISMATCH: max difference {max_diff:.3f}. The app is not processing audio like training — do not trust it.")

  25/100 pairs predicted (1s)
  50/100 pairs predicted (1s)
  75/100 pairs predicted (1s)
  100/100 pairs predicted (1s)


,sample_id,max |Δprob|,max |Δparam|,decision flips
pair,,,,
45,91-Mixture-Compressor-EQ-Reverb,0.001250,0.000119,0
6,9-Mixture-Compressor-EQ,0.000596,0.000120,0
91,52-Drums-Compressor-Reverb,0.000570,0.000180,0
38,25-Drums-Compressor,0.000535,0.000209,0
51,32-Bass-Reverb,0.000502,0.000147,0
22,4-Drums-EQ-Compressor-Reverb,0.000476,0.000242,0
94,90-Drums-EQ-Compressor-Reverb,0.000333,0.000165,0
19,4-Drums-Compressor-Reverb,0.000321,0.000222,0
36,90-Drums-EQ-Reverb-Compressor,0.000316,0.000176,0


✅ VERIFIED: app matches the Kaggle evaluation (max difference 1.25e-03).


## 8. Launch the Gradio app
Open the **`https://….gradio.live`** link printed below. Keep this Kaggle notebook **open and running** (interactive session, not *Save Version*) while you use the app — the link works only as long as this kernel is alive, and every launch prints a **new** link.

In [11]:
CHOICES = [(d["display"], d["pair"]) for d in DEMO]
BY_PAIR = {d["pair"]: d for d in DEMO}
import random as _random


def _is_correct(d):
    return RESULTS[d["pair"]]["chain12"] == d["targets"]["class12"]


def filtered_choices(cat, result):
    out = []
    for d in DEMO:
        if cat != "All" and d["source_category"] != cat:
            continue
        if result == "Correct only" and not _is_correct(d):
            continue
        if result == "Wrong only" and _is_correct(d):
            continue
        out.append((d["display"], d["pair"]))
    return out


def on_filter(cat, result):
    ch = filtered_choices(cat, result)
    note = f"{len(ch)} of {len(DEMO)} pairs match"
    return gr.update(choices=ch, value=ch[0][1] if ch else None), note


def on_step(pair, cat, result, step):
    ch = [v for _, v in filtered_choices(cat, result)]
    if not ch:
        return gr.update()
    i = ch.index(int(pair)) if pair is not None and int(pair) in ch else -1
    return gr.update(value=ch[(i + step) % len(ch)])


def on_random(cat, result):
    ch = [v for _, v in filtered_choices(cat, result)]
    return gr.update(value=_random.choice(ch)) if ch else gr.update()


def on_select(pair):
    if pair is None:
        return None, None, None, None, "", ""
    d = BY_PAIR[int(pair)]
    fig = spectrogram_figure(load_file_for_model(d["dry"]), load_file_for_model(d["wet"]), d["display"])
    return (d["dry_play"], d["wet_play"], fig, None,
            "<div style='padding:12px 0;color:var(--body-text-color-subdued)'>Press <b>Run model</b> to see the "
            "predicted chain, order and parameters next to the actual ones.</div>", "")


def on_run(pair):
    d = BY_PAIR[int(pair)]
    res = RESULTS.get(d["pair"]) or predict(load_file_for_model(d["dry"]), load_file_for_model(d["wet"]))
    return probs_for_label(res), result_view(res, d), result_markdown(res, d)


def param_score(res, d):
    """(close, compared, mean |err| 0-1) over parameters of effects in BOTH the predicted and actual chain."""
    ok = n = 0; errs = []
    for j in range(N_PARAMS):
        if d["param_mask"][j] and res["pred_presence"][PARAM_EFFECT_INDEX[j]]:
            _, good = fmt_error(j, res["params"][j], d["param_true"][j])
            ok += int(good); n += 1; errs.append(abs(float(res["params"][j]) - d["param_true"][j]))
    return ok, n, (float(np.mean(errs)) if errs else float("nan"))


def on_run_all():
    rows = []
    n_chain = n_pres = n_ord = n_ord_def = 0
    n_pok = n_pn = 0
    per_param = {j: [] for j in range(N_PARAMS)}
    for d in DEMO:
        res = RESULTS.get(d["pair"]) or predict(load_file_for_model(d["dry"]), load_file_for_model(d["wet"]))
        t = d["targets"]
        chain_ok = res["chain12"] == t["class12"]
        pres_ok = all(int(res["pred_presence"][k]) == int(t["presence"][k]) for k in range(3))
        defined = [j for j in range(2) if t["order_mask"][j]]
        ord_hits = [int(res["pred_order"][j]) == int(t["order"][j]) for j in defined]
        n_chain += chain_ok; n_pres += pres_ok; n_ord += sum(ord_hits); n_ord_def += len(defined)
        pok, pn, pmae = param_score(res, d); n_pok += pok; n_pn += pn
        for j in range(N_PARAMS):                       # every truly present effect (like the Kaggle test report)
            if d["param_mask"][j]:
                per_param[j].append((float(res["params"][j]), d["param_true"][j]))
        rows.append({"Pair": d["pair"], "Category": d["source_category"], "True (rendered)": " → ".join(d["chain"]),
                     "True (audible)": d["chain12_name"], "Predicted": res["chain_name"],
                     "p(pred)": round(res["chain_prob"], 3), "Chain": "✅" if chain_ok else "❌",
                     "Presence": "✅" if pres_ok else "❌",
                     "Order": ("✅" if all(ord_hits) else "❌") if defined else "—",
                     "Params close": f"{pok}/{pn}" if pn else "—",
                     "Param MAE": round(pmae, 3) if pn else None})
    n = len(DEMO)
    summary = (f"**{n} test pairs** · chain correct **{n_chain}/{n} ({n_chain / n:.0%})** · presence all-correct "
               f"{n_pres}/{n} · order questions correct {n_ord}/{n_ord_def} · parameters within tolerance "
               f"**{n_pok}/{n_pn} ({(n_pok / n_pn if n_pn else 0):.0%})** (effects in both chains)")
    df = pd.DataFrame(rows)
    per_chain = (df.assign(ok=df["Chain"] == "✅").groupby("True (audible)")["ok"].agg(["sum", "size"])
                 .reindex(CHAIN12_NAMES).dropna().astype(int).reset_index())
    per_chain.columns = ["True chain", "Correct", "Pairs"]
    per_chain["Accuracy"] = (per_chain["Correct"] / per_chain["Pairs"]).map("{:.0%}".format)
    prow = []
    for j, vals in per_param.items():
        if not vals:
            continue
        p, t = np.array(vals).T
        e = param_error(j, p, t)
        prow.append({"Parameter": PARAM_NAMES[j], "n": len(vals), "MAE (0-1)": round(float(np.mean(np.abs(p - t))), 4),
                     "Typical error": (f"{e['summary']:.2f} oct" if PARAM_SPEC[j][8] == "octave" else
                                       f"×{e['summary']:.2f}" if PARAM_SPEC[j][8] == "factor" else
                                       f"{e['summary']:.2f} {PARAM_SPEC[j][3]}".strip()),
                     "Within tolerance": f"{float(e['ok'].mean()):.0%}", "Tolerance": tolerance_text(j)})
    return df, summary, per_chain, pd.DataFrame(prow)


def on_upload(dry_path, wet_path):
    if not dry_path or not wet_path:
        return "Please upload **both** a dry and a wet file.", None, None, "", ""
    notes = []
    for name, p in (("Dry", dry_path), ("Wet", wet_path)):
        try:
            info = sf.info(p); sr, ch, dur = info.samplerate, info.channels, info.duration
        except Exception:
            y, sr = librosa.load(p, sr=None, mono=False); ch = 1 if y.ndim == 1 else y.shape[0]; dur = y.shape[-1] / sr
        if sr != SR: notes.append(f"{name}: {sr} Hz → resampled to {SR} Hz")
        if ch > 1: notes.append(f"{name}: {ch} channels → mixed to mono")
        if dur > CFG["AUDIO_LENGTH_SECONDS"] + 0.01: notes.append(f"{name}: {dur:.1f} s → only the first {CFG['AUDIO_LENGTH_SECONDS']:.0f} s is used")
        if dur < CFG["AUDIO_LENGTH_SECONDS"] - 0.01: notes.append(f"{name}: {dur:.1f} s → zero-padded to {CFG['AUDIO_LENGTH_SECONDS']:.0f} s")
        if name == "Dry": dry_dur = dur
        else: wet_dur = dur
    if abs(dry_dur - wet_dur) > 0.05:
        notes.append(f"Dry ({dry_dur:.2f} s) and wet ({wet_dur:.2f} s) differ in length — they must be the same, time-aligned recording")
    dry, wet = load_file_for_model(dry_path), load_file_for_model(wet_path)
    res = predict(dry, wet)
    warn = ("**Notes:**\n" + "\n".join(f"- {x}" for x in notes) + "\n\n" if notes else "") + \
           "_The model was trained on Pedalboard EQ/Compressor/Reverb applied to 10 s MUSDB-style stems; " \
           "other audio or other plugins may give unreliable results._"
    return warn, spectrogram_figure(dry, wet, "Uploaded pair"), probs_for_label(res), result_view(res), result_markdown(res)


with gr.Blocks(title="Inverse FX Predictor — type, order, parameters") as demo:
    gr.Markdown(f"## Inverse FX Predictor — which effects, in which order, with which settings?\n"
                f"Model `{MODEL_REPO}` (three-head Log-Mel Transformer, best epoch {state['best_epoch']}). "
                f"Inputs are only the dry and wet audio. The output is the most probable of the 12 audible chains "
                f"(EQ↔Reverb order is not audible, so those renders share one chain).")
    with gr.Tabs():
        with gr.Tab("Test pairs"):
            with gr.Row():
                cat_filter = gr.Radio(["All", "Bass", "Drums", "Mixture"], value="All", label="Source type")
                res_filter = gr.Radio(["All", "Correct only", "Wrong only"], value="All", label="Model result (chain)")
                filter_note = gr.Markdown(f"{len(DEMO)} of {len(DEMO)} pairs match")
            with gr.Row():
                prev_btn = gr.Button("◀ Prev", scale=0, min_width=90)
                pick = gr.Dropdown(choices=CHOICES, value=CHOICES[0][1], label="Test pair (true answer hidden)", scale=4)
                next_btn = gr.Button("Next ▶", scale=0, min_width=90)
                rand_btn = gr.Button("🎲 Random", scale=0, min_width=110)
                run_btn = gr.Button("Run model", variant="primary", scale=1)
            with gr.Row():
                dry_player = gr.Audio(label="Dry (input)", type="filepath", interactive=False)
                wet_player = gr.Audio(label="Wet (processed)", type="filepath", interactive=False)
            result_card = gr.HTML()
            with gr.Row():
                with gr.Column(scale=1):
                    prob_label = gr.Label(label="Top 5 most probable chains", num_top_classes=5)
                with gr.Column(scale=2):
                    with gr.Accordion("Detailed tables", open=False):
                        result_md = gr.Markdown()
            with gr.Accordion("Log-mel spectrograms (dry, wet, wet − dry)", open=False):
                spec_plot = gr.Plot(label="Log-mel spectrograms")
            with gr.Accordion("Run all demo pairs", open=False):
                all_btn = gr.Button("Run all")
                all_summary = gr.Markdown()
                all_per_chain = gr.Dataframe(interactive=False, label="Accuracy per true chain")
                all_per_param = gr.Dataframe(interactive=False, label="Parameter error per parameter (effects really present)")
                all_table = gr.Dataframe(interactive=False, wrap=True, label="All pairs", max_height=500)
        with gr.Tab("Upload your own"):
            with gr.Row():
                up_dry = gr.Audio(label="Dry audio", type="filepath")
                up_wet = gr.Audio(label="Wet audio", type="filepath")
            up_btn = gr.Button("Predict effect chain + parameters", variant="primary")
            up_notes = gr.Markdown()
            up_card = gr.HTML()
            with gr.Row():
                with gr.Column(scale=1):
                    up_label = gr.Label(label="Top 5 most probable chains", num_top_classes=5)
                with gr.Column(scale=2):
                    with gr.Accordion("Detailed tables", open=False):
                        up_md = gr.Markdown()
            with gr.Accordion("Log-mel spectrograms (dry, wet, wet − dry)", open=False):
                up_plot = gr.Plot(label="Log-mel spectrograms")

    sel_out = [dry_player, wet_player, spec_plot, prob_label, result_card, result_md]
    pick.change(on_select, inputs=pick, outputs=sel_out)
    demo.load(on_select, inputs=pick, outputs=sel_out)
    run_btn.click(on_run, inputs=pick, outputs=[prob_label, result_card, result_md])
    all_btn.click(on_run_all, outputs=[all_table, all_summary, all_per_chain, all_per_param])
    for f in (cat_filter, res_filter):
        f.change(on_filter, inputs=[cat_filter, res_filter], outputs=[pick, filter_note])
    prev_btn.click(lambda p, c, r: on_step(p, c, r, -1), inputs=[pick, cat_filter, res_filter], outputs=pick)
    next_btn.click(lambda p, c, r: on_step(p, c, r, +1), inputs=[pick, cat_filter, res_filter], outputs=pick)
    rand_btn.click(on_random, inputs=[cat_filter, res_filter], outputs=pick)
    up_btn.click(on_upload, inputs=[up_dry, up_wet], outputs=[up_notes, up_plot, up_label, up_card, up_md])

demo.queue(default_concurrency_limit=2).launch(share=True, inline=False, allowed_paths=[WORK], show_error=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://c3c0538f79ece42ac1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 9. Stop the app (only when you are done)
`launch()` does not block in a notebook, so with *Run All* this cell would run right after Section 8 and shut the app down. It therefore does nothing unless you set `STOP_APP = True` and run it yourself.

In [12]:
STOP_APP = False          # set True and run this cell to stop the app
if STOP_APP:
    demo.close()
    print("Gradio app stopped.")
else:
    print("App is still running — open the gradio.live link from Section 8. Set STOP_APP = True and re-run this cell to stop it.")

App is still running — open the gradio.live link from Section 8. Set STOP_APP = True and re-run this cell to stop it.
